# _Python for Scientific Data Analysis_


#  AstroPy and AstroQuery

## Section 3: AstroQuery

## (_Note_ : Outages Affecting Astroquery) 

Currently -- 11/17/2026 -- CDS is experiencing a major outage.  It has wrecked havoc on my notes: fixing the notes has taken some time. The entire astroquery infrastructure is affected: I include information about mirror sites as a result.  It is possible that further outages will occur: please save a copy of this notebook so you can see the output when the queries are working.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import astroquery

from astropy.coordinates import Angle

In [2]:
from astroquery.simbad import Simbad #Simbad astronomical database
from astroquery.vizier import Vizier #Vizier database

### Vizier


The simplest form of a Vizier query involves searching for data for a given object within a catalogue, searchable either on the object name itself or around its coordinates.  Here are some examples [made more complicated by the CDS outages]

In [3]:
#stupid stuff related to outages
Vizier.clear_cache()
Vizier.VIZIER_SERVER = 'vizier.cfa.harvard.edu'
#Vizier.VIZIER_SERVER = 'vizier.nao.ac.jp'
Vizier.TIMEOUT = 20

v=Vizier(columns=['**'],vizier_server='vizier.cfa.harvard.edu') #to work around yet another outage/crash: innocuous when CDS is behaving
         
#v=Vizier(columns=['**'],vizier_server='vizier.idia.ac.za')
#v=Vizier(columns=['**'],vizier_server='vizier.nao.ac.jp')
#v.TIMEOUT=120

#catalogs: these are the Hipparcos and 2MASS catalogs
catname=['I/239/hip_main','II/246/out']

#names of 3 stars

targlists=["HR 8799","HIP 99770","AF Lep"]

for t in targlists:

 try:
  result=v.query_object(t,cache=False,catalog=catname)
  print(t,'->',len(result),' tables')
 except Exception as e: 
  print('failed on {0:s}: {1} {2}'.format(t,type(e).__name__,str(e)[:200]))

v.query_object('HIP 99770', catalog='II/246/out', radius=Angle(20, 'arcsec'), cache=False)[0]['_2MASS', 'Jmag', 'Hmag', 'Kmag']

HR 8799 -> 2  tables
HIP 99770 -> 2  tables
AF Lep -> 2  tables


_2MASS,Jmag,Hmag,Kmag
,mag,mag,mag
str17,float32,float32,float32
20143203+3648225,4.886,4.688,4.422
20143054+3648274,12.677,12.179,12.087


In [4]:
v = Vizier(columns=['**'], vizier_server='vizier.cfa.harvard.edu', timeout=20)
t = v.query_object('HR 8799', catalog='II/246/out', cache=False)[0]

print('the column names are ',t.colnames)
#pulling specific rows out of the table queried by Vizier: offset from search position then RA,DEC, 2MASS name, J, H, K, quality flag
t_rowselect = t['_r', 'RAJ2000', 'DEJ2000', '_2MASS', 'Jmag', 'e_Jmag', 'Hmag', 'e_Hmag', 'Kmag', 'e_Kmag', 'Qflg']

t_rowselect

the column names are  ['_r', 'RAJ2000', 'DEJ2000', 'errMaj', 'errMin', 'errPA', '_2MASS', 'Jmag', 'Jcmsig', 'e_Jmag', 'Jsnr', 'Hmag', 'Hcmsig', 'e_Hmag', 'Hsnr', 'Kmag', 'Kcmsig', 'e_Kmag', 'Ksnr', 'Qflg', 'Rflg', 'Bflg', 'Cflg', 'Ndet', 'prox', 'pxPA', 'pxCntr', 'Xflg', 'Aflg', 'Cntr', 'Hemis', 'Date', 'Scan', 'GLON', 'GLAT', 'Xscan', 'JD', 'Jpsfchi', 'Hpsfchi', 'Kpsfchi', 'Jstdap', 'e_Jstdap', 'Hstdap', 'e_Hstdap', 'Kstdap', 'e_Kstdap', 'edgeNS', 'edgeEW', 'edge', 'dup', 'use', 'opt', 'Dopt', 'PAopt', 'Bmag', 'Rmag', 'Nopt', 'extKey', 'scanKey', 'coaddKey', 'coadd', 'Opt']


_r,RAJ2000,DEJ2000,_2MASS,Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,Qflg
,deg,deg,,mag,mag,mag,mag,mag,mag,
float64,float64,float64,str17,float32,float32,float32,float32,float32,float32,str3
1.3857,346.869453,21.111158,23072866+2106401,11.678,--,12.031,--,9.415,0.146,UUB
0.9672,346.852378,21.133661,23072457+2108011,7.574,0.020,6.975,0.033,6.843,0.020,AAA
0.0058,346.869545,21.134260,23072869+2108033,5.383,0.027,5.280,0.018,5.240,0.018,AAA
1.3576,346.856797,21.153444,23072563+2109123,15.083,0.049,14.666,0.062,14.590,0.107,AAA


In [5]:
#result4=Vizier.query_object("HIP 99770",catalog='I/239/tyc_main')
result4=Vizier.query_object("HIP 99770",catalog='I/259/tyc2')

print(result4 ) #what is this table list?
print(len(result4)) #how many entries does it have?
result4[0]

TableList with 1 tables:
	'0:I/259/tyc2' with 10 column(s) and 1 row(s) 
1


TYC1,TYC2,TYC3,pmRA,pmDE,BTmag,VTmag,HIP,RA_ICRS_,DE_ICRS_
,,,mas / yr,mas / yr,mag,mag,,deg,deg
int16,int16,uint8,float32,float32,float32,float32,int32,float64,float64
2683,3962,1,68.9,71.8,5.121,4.962,99770,303.63326583,36.80614139


If we want to print out the table entries we can do that here by accessing the relevant subscript of the table.  With the above print statements we learn that this table has only one entry (which is good! There isn't a duplicate HIP 99770 entry floating around this giant catalogue).

See below

In [6]:
result4[0] #return the first (and only) table

TYC1,TYC2,TYC3,pmRA,pmDE,BTmag,VTmag,HIP,RA_ICRS_,DE_ICRS_
,,,mas / yr,mas / yr,mag,mag,,deg,deg
int16,int16,uint8,float32,float32,float32,float32,int32,float64,float64
2683,3962,1,68.9,71.8,5.121,4.962,99770,303.63326583,36.80614139


Now, the above command just simply selected the first table in the table list.  You can query multiple tables at once: you just have to enclose them in square brackets...

So we are going to query a different Tycho catalog (first entry), the Hipparcos catalog (second entry), and the 2MASS catalog (third entry).  These are all all-sky surveys so they have data for a ton of objects.

In [7]:
#CAUTION!!! - note the addition of the "Angle" keyword here
from astropy.coordinates import Angle

starname='HIP 54515'
result4b=Vizier.query_object(starname,catalog=['I/239/tyc_main','I/239/hip_main','II/246/out'],radius=Angle(5,"arcsec"),cache=False)
#                              radius=Angle(1,"deg"))
#result4b
result4b #now return the second of the tables

result4b[2]

result4b[2]['RAJ2000','DEJ2000','_2MASS','Jmag','e_Jmag','Hmag','Kmag','Qflg']

RAJ2000,DEJ2000,_2MASS,Jmag,e_Jmag,Hmag,Kmag,Qflg
deg,deg,,mag,mag,mag,mag,
float64,float64,str17,float32,float32,float32,float32,str3
167.306933,-1.333231,11091366-0119596,6.510,0.023,6.464,6.422,AAA


With this subtlety (Table _list_ vs Table), we can now proceed to return entry values in specific tables.  E.g. ...

In [8]:
#result4[0].colnames #lists the column names
#result4[0]['TYC'] #returns the column name & all entries in the 'TYC' column
name_tyc=result4b[0]['TYC'][0] #returns the entry for this star
ra_tyc=result4b[0]['RAhms'][0]
vmag=result4b[0]['Vmag'][0]

print(name_tyc)
print(ra_tyc) #note: this is a string as you can see from the table metadata
print(vmag) #floating point

4921  1206 1
11 09 13.70
6.82


In [9]:
result4b[2] #now let's inspect the 3rd table

RAJ2000,DEJ2000,_2MASS,Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,Qflg,Rflg,Bflg,Cflg,Xflg,Aflg
deg,deg,,mag,mag,mag,mag,mag,mag,,,,,,
float64,float64,str17,float32,float32,float32,float32,float32,float32,str3,str3,str3,str3,uint8,uint8
167.306933,-1.333231,11091366-0119596,6.510,0.023,6.464,0.040,6.422,0.021,AAA,111,111,000,0,0


In [10]:
result4b[2]['Jmag'][0] #return the J band magnitude. 

6.51



Arguably the greater power with catalogue queries is searching entire catalogues on condition or querying around a position (any position: doesn't have to be centered on a star).    You can do this with the ``query_region`` function call.


In [11]:
#from astroquery.vizier import Vizier #already imported this
#from astroquery.coordinates import Angle #already imported this

result5=Vizier.query_region("HIP 99770",radius=Angle(0.15,"deg"),catalog='II/246/out')
result5[0]

RAJ2000,DEJ2000,_2MASS,Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,Qflg,Rflg,Bflg,Cflg,Xflg,Aflg
deg,deg,,mag,mag,mag,mag,mag,mag,,,,,,
float64,float64,str17,float32,float32,float32,float32,float32,float32,str3,str3,str3,str3,uint8,uint8
303.686016,36.664169,20144464+3639510,15.489,0.067,14.945,0.103,14.477,--,AAU,226,220,cc0,0,0
303.681316,36.675674,20144351+3640324,16.542,--,15.090,0.095,14.286,0.091,UAA,22,11,000,0,0
303.685663,36.670895,20144455+3640152,15.872,0.083,15.001,0.090,14.879,0.145,AAB,222,111,000,0,0
303.685310,36.665012,20144447+3639540,15.758,0.072,15.167,0.102,14.488,--,AAU,226,220,cc0,0,0
303.676842,36.668884,20144244+3640079,15.851,0.081,15.286,0.113,14.924,--,ABU,220,110,000,0,0
303.680029,36.666634,20144320+3639598,15.014,0.039,13.590,0.027,13.065,0.030,AAA,222,111,000,0,0
303.679284,36.671806,20144302+3640185,16.071,--,14.532,0.056,13.748,0.055,UAA,22,11,0cc,0,0
303.696010,36.666199,20144704+3639583,14.298,0.027,13.945,0.036,13.897,0.062,AAA,222,111,000,0,0


Now, if you look at this table you and you know anything about the star we are querying around, you immediately see a problem. Namely, we do not see our star listed (it should be about magnitude 4.5 in each 2MASS passband).

You also see the Table size is 50 (50 entries).   If you make the search radius larger you _still_ get 50 entries. 

What's happening here is that a) there are a lot of objects within our search radius and b) astroquery is truncating the entries to 50.  

So we must modify our querying ...

First, one thing we can do is tell astroquery to return _all_ row values

In [12]:
Vizier.ROW_LIMIT = -1   #all values

#Vizier.ROW_LIMIT = 50 #if we want to restore the default
#Vizier.ROW_LIMIT = 100 #if we want to return 100 values

result5b=Vizier.query_region("HIP 99770",radius=Angle(0.15,"deg"),catalog='II/246/out')
result5b[0]

RAJ2000,DEJ2000,_2MASS,Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,Qflg,Rflg,Bflg,Cflg,Xflg,Aflg
deg,deg,,mag,mag,mag,mag,mag,mag,,,,,,
float64,float64,str17,float32,float32,float32,float32,float32,float32,str3,str3,str3,str3,uint8,uint8
303.686016,36.664169,20144464+3639510,15.489,0.067,14.945,0.103,14.477,--,AAU,226,220,cc0,0,0
303.681316,36.675674,20144351+3640324,16.542,--,15.090,0.095,14.286,0.091,UAA,22,11,000,0,0
303.685663,36.670895,20144455+3640152,15.872,0.083,15.001,0.090,14.879,0.145,AAB,222,111,000,0,0
303.685310,36.665012,20144447+3639540,15.758,0.072,15.167,0.102,14.488,--,AAU,226,220,cc0,0,0
303.676842,36.668884,20144244+3640079,15.851,0.081,15.286,0.113,14.924,--,ABU,220,110,000,0,0
303.680029,36.666634,20144320+3639598,15.014,0.039,13.590,0.027,13.065,0.030,AAA,222,111,000,0,0
303.679284,36.671806,20144302+3640185,16.071,--,14.532,0.056,13.748,0.055,UAA,22,11,0cc,0,0
303.696010,36.666199,20144704+3639583,14.298,0.027,13.945,0.036,13.897,0.062,AAA,222,111,000,0,0


We can also return a _sorted_ table by adding a ``+`` or ``-`` in front of column names.   Let's sort on the J band magnitude.

In [13]:
#note * means "standard columns" ... otherwise you would only return the Jmag column

#again, outage madness reqs you to set the server
v2=Vizier(columns=["*","+Jmag"],vizier_server='vizier.cfa.harvard.edu') 
v2.ROW_LIMIT = -1

results5c=v2.query_region("HIP 99770",radius=Angle(0.15,"deg"),catalog="II/246/out")
print(results5c[0])

 RAJ2000    DEJ2000        _2MASS       Jmag  e_Jmag ... Bflg Cflg Xflg Aflg
   deg        deg                       mag    mag   ...                    
---------- ---------- ---------------- ------ ------ ... ---- ---- ---- ----
303.665097  36.752087 20143962+3645075  4.552  0.276 ...  111  000    0    0
303.633484  36.806255 20143203+3648225  4.886  0.306 ...  111  000    0    0
303.799735  36.863590 20151193+3651489  6.802  0.023 ...  111  000    0    0
303.632409  36.661018 20143177+3639396  6.980  0.027 ...  111  000    0    0
303.801214  36.769802 20151229+3646112  7.551  0.018 ...  111  000    0    0
303.638422  36.915199 20143322+3654547  8.341  0.020 ...  111  000    0    0
303.730587  36.722252 20145534+3643201  8.412  0.019 ...  111  000    0    0
303.687070  36.691135 20144489+3641280  8.585  0.018 ...  111  000    0    0
303.501693  36.779751 20140040+3646471  8.611  0.027 ...  111  000    0    0
       ...        ...              ...    ...    ... ...  ...  ...  ...  ...

Now, it's not entirely clear which entry is the one we want from this list just from looking at it.  Is it the first or second entry?

If we compare with the Simbad position output we see that it is the second.

In [14]:
#outages require a fix
#Simbad.server='simbad.harvard.edu' #for astroquery 0.4.11 this works
Simbad.SIMBAD_URL = 'https://simbad.harvard.edu/simbad/sim-script' #for astroquery 0.4.7 you need to do this

starcoords=Simbad.query_object("HIP 99770")
print(starcoords[0]['RA'],starcoords[0]['DEC'])

20 14 32.0323 +36 48 22.700


Finally, the most obvious thing to do is to just shrink our search radius.  If you do so, you (naturally) return fewer stars.  So let's shrink the radius to 20 arcseconds

In [15]:
v3=Vizier(columns=['**'],vizier_server='vizier.cfa.harvard.edu')
result5d=v3.query_region("HIP 99770",radius=Angle(20,"arcsec"),catalog='II/246/out')
result5d[0]

_r,RAJ2000,DEJ2000,errMaj,errMin,errPA,_2MASS,Jmag,Jcmsig,e_Jmag,Jsnr,Hmag,Hcmsig,e_Hmag,Hsnr,Kmag,Kcmsig,e_Kmag,Ksnr,Qflg,Rflg,Bflg,Cflg,Ndet,prox,pxPA,pxCntr,Xflg,Aflg,Cntr,Hemis,Date,Scan,GLON,GLAT,Xscan,JD,Jpsfchi,Hpsfchi,Kpsfchi,Jstdap,e_Jstdap,Hstdap,e_Hstdap,Kstdap,e_Kstdap,edgeNS,edgeEW,edge,dup,use,opt,Dopt,PAopt,Bmag,Rmag,Nopt,extKey,scanKey,coaddKey,coadd,Opt
,deg,deg,arcsec,arcsec,deg,,mag,mag,mag,,mag,mag,mag,,mag,mag,mag,,,,,,,arcsec,deg,,,,,,,,deg,deg,arcsec,d,,,,mag,mag,mag,mag,mag,mag,arcsec,arcsec,,,,,arcsec,deg,mag,mag,,,,,,
float64,float64,float64,float32,float32,int16,str17,float32,float32,float32,float64,float32,float32,float32,float64,float32,float32,float32,float64,str3,str3,str3,str3,str6,float32,int16,int64,uint8,uint8,int32,str1,str10,int16,float64,float64,float32,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32,int16,str2,uint8,uint8,str1,float32,int16,float32,float32,uint8,int32,int32,int32,int16,str3
0.206,303.633484,36.806255,0.11,0.11,0,20143203+3648225,4.886,0.306,0.306,337311.0,4.688,0.242,0.242,167382.0,4.422,0.009,0.017,109049.0,DDA,331,111,000,60666,18.6,285,304838469,0,0,304838495,n,1998-06-21,49,74.448,1.172,169.1,2450985.7917,--,--,--,--,--,--,--,--,--,3429,84,sw,0,1,T,0.2,117,5.08,4.95,2,--,19222,442103,232,Opt
18.539,303.627252,36.807613,0.11,0.11,173,20143054+3648274,12.677,--,--,--,12.179,--,--,--,12.087,0.045,0.046,93.7,UUA,2,2,00c,66,11.5,288,304838456,0,0,304838469,n,1998-06-21,49,74.446,1.176,187.1,2450985.7917,--,--,3.20,--,--,--,--,11.924,0.025,3434,66,sw,1,1,0,--,--,--,--,0,--,19222,442103,232,Opt


_Now_ it is clearer which entry we want.

We also could have kept the search radius the same but added _column filters_ to downselect on objects that "might" be the one(s) we are interested in.   See below ...

In [16]:
result5e=v3.query_region("HIP 99770",radius=Angle(0.15,"deg"),catalog='II/246/out',column_filters={"Jmag":"<6"})
print(result5e[0])

   _r     RAJ2000    DEJ2000   errMaj errMin ... scanKey coaddKey coadd Opt
            deg        deg     arcsec arcsec ...                           
-------- ---------- ---------- ------ ------ ... ------- -------- ----- ---
0.059848 303.665097  36.752087   0.29   0.29 ...   19222   442103   232 Opt
0.000057 303.633484  36.806255   0.11   0.11 ...   19222   442103   232 Opt


In [17]:
#slightly more complex

#the two dots define a range
result5f=v3.query_region("HIP 99770",radius=Angle(0.15,"deg"),catalog='II/246/out',
                         column_filters={"Jmag":"4 .. 9.7"})
print(result5f[0])

   _r     RAJ2000    DEJ2000   errMaj errMin ... scanKey coaddKey coadd Opt
            deg        deg     arcsec arcsec ...                           
-------- ---------- ---------- ------ ------ ... ------- -------- ----- ---
0.135178 303.639025  36.671207   0.07   0.06 ...   19222   442103   232 Opt
0.145296 303.632409  36.661018   0.07   0.06 ...   19222   442103   232 Opt
0.136586 303.744294  36.702522   0.07   0.06 ...   19222   442103   232 Opt
0.114535 303.730587  36.722252   0.07   0.06 ...   19222   442103   232 Opt
0.122922 303.687070  36.691135   0.07   0.06 ...   19222   442103   232 Opt
0.123967 303.784785  36.780125   0.06   0.06 ...   19223   442110    44 Opt
0.139208 303.801214  36.769802   0.06   0.06 ...   19223   442110    44 Opt
0.101516 303.570011  36.718445   0.06   0.06 ...   40294   926759   232 Opt
0.059848 303.665097  36.752087   0.29   0.29 ...   19222   442103   232 Opt
     ...        ...        ...    ...    ... ...     ...      ...   ... ...
0.136296 303

Now, we don't actually have to center the search on the position of a known object.  We can center it on blank space, provided that put in a coordinate object in our Vizier call.   Here's what I mean ...

In [18]:
from astropy.coordinates import SkyCoord
#let's select a region just slightly off HIP 99770, about halfway between it and the other bright star
blankspace=SkyCoord('20h14m37.0s','+36d46m00.0s')
blankspace

<SkyCoord (ICRS): (ra, dec) in deg
    (303.65416667, 36.76666667)>

In [19]:
v2=Vizier(columns=["*","+Jmag"],vizier_server='vizier.cfa.harvard.edu')

result6=v2.query_region(blankspace,
   radius=Angle(1,"deg"),catalog='II/246/out',column_filters={"Jmag":"<18"})

result6[0]

RAJ2000,DEJ2000,_2MASS,Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,Qflg,Rflg,Bflg,Cflg,Xflg,Aflg
deg,deg,,mag,mag,mag,mag,mag,mag,,,,,,
float64,float64,str17,float32,float32,float32,float32,float32,float32,str3,str3,str3,str3,uint8,uint8
303.665097,36.752087,20143962+3645075,4.552,0.276,3.796,0.228,3.497,0.382,DDD,333,111,000,0,0
302.781041,36.130852,20110744+3607510,4.559,0.242,3.524,0.238,2.869,0.386,DDD,333,111,000,0,0
302.736799,36.579788,20105683+3634472,4.688,0.274,3.532,0.226,2.775,0.348,DDD,333,111,000,0,0
304.849469,36.729591,20192387+3643465,4.836,0.186,3.567,0.192,2.821,0.276,CCD,333,111,000,0,0
303.633484,36.806255,20143203+3648225,4.886,0.306,4.688,0.242,4.422,0.017,DDA,331,111,000,0,0
302.775625,36.113567,20110615+3606488,5.033,0.017,3.884,0.276,2.777,0.386,ADD,133,111,000,0,0
303.057928,35.979988,20121390+3558479,5.050,0.306,3.993,0.242,3.449,0.436,DDD,333,111,000,0,0
303.527681,36.278870,20140664+3616439,5.141,0.026,4.475,0.240,4.179,0.020,ADA,131,111,000,0,0
